In [27]:
import numpy as np
import pandas as pd

exp = pd.read_csv('../data_raw/Exp-DB.csv')
mix = pd.read_csv('../data_raw/MixSolDB.csv')

C:\Users\cassi\AppData\Local\Temp\ipykernel_14840\272824951.py:5: DtypeWarning: Columns (12: density_gml-1) have mixed types. Specify dtype option on import or set low_memory=False.
  mix = pd.read_csv('../data_raw/MixSolDB.csv')


In [28]:
exp.shape

(23996, 11)

In [29]:
mix.shape

(56789, 16)

In [30]:
exp.columns

Index(['solute', 'solvent', 'logS', 'DGsolv', 'Reference (DOI)', 'Comments',
       'can_smiles_solute', 'can_smiles_solvent', 'concat_smiles',
       'is_chonps_halogens', 'index'],
      dtype='str')

In [31]:
mix.columns

Index(['can_smiles_solute', 'can_smiles_solvent1', 'can_smiles_solvent2',
       'can_smiles_solvent3', 'DGsolv', 'DGsolv_converted', 'logS', 'logXs',
       'mol_frac_solvent1', 'mol_frac_solvent2', 'mol_frac_solvent3', 'T_K',
       'density_gml-1', 'vp_atm', 'vp_pred_atm', 'db_source'],
      dtype='str')

In [32]:
def clean_doi(doi):
    if pd.isna(doi):
        return None

    return doi.split(" (")[0].strip()

lookup = {}

for j in range(len(mix)):
    doi = mix.iloc[j, -1]

    cleaned = clean_doi(doi)

    if cleaned is None:
        continue

    if cleaned in lookup:
        lookup[cleaned].append(j)
    else:
        lookup[cleaned] = [j]

print(len(lookup))
print(sum(len(positions) for positions in lookup.values()))

134
56789


In [33]:
multirow_sources = {
    doi: positions
    for doi, positions in lookup.items()
    if len(positions) > 1
}

print(len(multirow_sources))

111


In [34]:
exp_compare = exp.copy()
mix_compare = mix.copy()

In [35]:
exp_compare["clean_doi"] = exp_compare["Reference (DOI)"].map(clean_doi)

mix_compare["clean_doi"] = mix_compare["db_source"].map(clean_doi)

In [36]:
exp_compare = exp_compare.reset_index(names="exp_index")
mix_compare = mix_compare.reset_index(names="mix_index")

In [37]:
pair_check = exp_compare.merge(
    mix_compare,
    left_on=[
        "clean_doi",
        "can_smiles_solute",
        "can_smiles_solvent"
    ],
    right_on=[
        "clean_doi",
        "can_smiles_solute",
        "can_smiles_solvent1"
    ],
    how="left",
    suffixes=("_exp", "_mix"),
    indicator=True
)

In [38]:
pair_check["_merge"].value_counts()

_merge
both          24270
left_only      1230
right_only        0
Name: count, dtype: int64

In [39]:
matched_exp = pair_check.loc[
    pair_check["_merge"] == "both",
    "exp_index"
].nunique()

print("Unique Exp rows with a candidate match:", matched_exp)
print("Unique Exp rows with no candidate match:", len(exp) - matched_exp)

Unique Exp rows with a candidate match: 22766
Unique Exp rows with no candidate match: 1230


In [40]:
pair_check[
    [
        "exp_index",
        "mix_index",
        "DGsolv_exp",
        "DGsolv_mix",
        "logS_exp",
        "logS_mix",
        "_merge"
    ]
].head(20)

,exp_index,mix_index,DGsolv_exp,DGsolv_mix,logS_exp,logS_mix,_merge
0,0,8281.0,NaN,NaN,-6.390,-6.390,both
1,1,8282.0,NaN,NaN,-4.740,-4.740,both
2,2,8283.0,-1.23,-1.23,NaN,NaN,both
3,3,8284.0,-1.19,-1.19,NaN,NaN,both
4,4,8285.0,-1.11,-1.11,NaN,NaN,both
5,5,8286.0,-1.09,-1.09,NaN,NaN,both
6,6,8287.0,NaN,NaN,-1.740,-1.740,both
7,7,8288.0,NaN,NaN,-1.320,-1.320,both
8,8,8289.0,-4.72,-4.72,NaN,NaN,both
9,9,8290.0,NaN,NaN,-3.140,-3.140,both


For this next part we check to see if DGsolv matches for each row tagged with 'both'. Then we do the same with LogS

In [41]:
pair_check["DGsolv_match"] = (
    pair_check["DGsolv_exp"].notna()
    & pair_check["DGsolv_mix"].notna()
    & np.isclose(
        pair_check["DGsolv_exp"],
        pair_check["DGsolv_mix"],
        atol=1e-8,
        rtol=0
    )
)

In [42]:
pair_check["logS_match"] = (
    pair_check["logS_exp"].notna()
    & pair_check["logS_mix"].notna()
    & np.isclose(
        pair_check["logS_exp"],
        pair_check["logS_mix"],
        atol=1e-8,
        rtol=0
    )
)

In [43]:
pair_check["value_match"] = (
    pair_check["DGsolv_match"]
    | pair_check["logS_match"]
)

Now we need to classify the original Exp index numbers

In [44]:
candidate_indices = set(
    pair_check.loc[
        pair_check["_merge"] == "both",
        "exp_index"
    ]
)

In [45]:
confirmed_indices = set(
    pair_check.loc[
        pair_check["value_match"],
        "exp_index"
    ]
)

In [46]:
#  Exp rows that werent included in Mix
pair_only_indices = candidate_indices - confirmed_indices

# Exp rows that never matched DOI + solute + solvent in the first place
no_pair_indices = set(exp.index) - candidate_indices

print("Confirmed already in MixSol:", len(confirmed_indices))
print("Same DOI + pair, but value differs:", len(pair_only_indices))
print("No DOI + pair match:", len(no_pair_indices))

Confirmed already in MixSol: 22680
Same DOI + pair, but value differs: 86
No DOI + pair match: 1230


Investigating difference between Exp rows and MixSol rows that have the same doi, solvent, and solute but have different S values, likely due to MixSol's multi-component nature.

In [47]:
value_review = pair_check[
    pair_check["exp_index"].isin(pair_only_indices)
].copy()

In [48]:
value_review[
    [
        "exp_index",
        "mix_index",
        "Reference (DOI)",
        "can_smiles_solute",
        "can_smiles_solvent",
        "DGsolv_exp",
        "DGsolv_mix",
        "logS_exp",
        "logS_mix",
        "T_K"
    ]
].head(20)

,exp_index,mix_index,Reference (DOI),can_smiles_solute,can_smiles_solvent,DGsolv_exp,DGsolv_mix,logS_exp,logS_mix,T_K
2936,2936,10988.0,10.1038/npre.2010.4243.3,CC(=O)Oc1ccccc1C(=O)O,ClC(Cl)Cl,NaN,NaN,-1.71,-0.744727,298.15
3577,3577,21.0,10.1016/s0304-3894(01)00312-0,CC(C)C(C(=O)OC(C#N)c1cccc(Oc2ccccc2)c1)c1ccc(C...,CO,NaN,NaN,-0.32,-0.322543,298.15
3578,3577,31.0,10.1016/s0304-3894(01)00312-0,CC(C)C(C(=O)OC(C#N)c1cccc(Oc2ccccc2)c1)c1ccc(C...,CO,NaN,NaN,-0.32,-0.286851,303.15
3579,3577,41.0,10.1016/s0304-3894(01)00312-0,CC(C)C(C(=O)OC(C#N)c1cccc(Oc2ccccc2)c1)c1ccc(C...,CO,NaN,NaN,-0.32,-0.231484,308.15
4259,4257,7981.0,10.1007/s10953-007-9228-0,CC(C)Cc1ccc(C(C)C(=O)O)cc1,CC(O)CO,NaN,NaN,0.05,-0.047692,293.15
4260,4257,7987.0,10.1007/s10953-007-9228-0,CC(C)Cc1ccc(C(C)C(=O)O)cc1,CC(O)CO,NaN,NaN,0.05,0.054230,298.15
4261,4257,7993.0,10.1007/s10953-007-9228-0,CC(C)Cc1ccc(C(C)C(=O)O)cc1,CC(O)CO,NaN,NaN,0.05,0.181272,303.15
4262,4257,7999.0,10.1007/s10953-007-9228-0,CC(C)Cc1ccc(C(C)C(=O)O)cc1,CC(O)CO,NaN,NaN,0.05,0.277609,308.15
4263,4257,8005.0,10.1007/s10953-007-9228-0,CC(C)Cc1ccc(C(C)C(=O)O)cc1,CC(O)CO,NaN,NaN,0.05,0.321598,313.15
9785,9779,3691.0,10.1021/acs.jced.0c00371,CCCCS(=O)(=O)N[C@@H](Cc1ccc(OCCCCC2CCNCC2)cc1)...,CC(C)O,NaN,NaN,-2.69,-2.994727,278.15


In [50]:
value_review["DGsolv_difference"] = (
    value_review["DGsolv_exp"]
    - value_review["DGsolv_mix"]
)

value_review["logS_difference"] = (
    value_review["logS_exp"]
    - value_review["logS_mix"]
)

print(len(value_review))
print(value_review["exp_index"].nunique())

1586
86


In [51]:
mix_single = mix[
    mix["can_smiles_solvent2"].isna()
    & mix["can_smiles_solvent3"].isna()
].copy()